# AdaptAuth-ML · Experimento S8, versión 5 (objetivo F1 ≥ 0.60)

**Desde el celular:** **Entorno de ejecución → Ejecutar todo** y acepta el permiso de Google Drive.

| Corrida | Qué hace | Tiempo aprox. |
|---|---|---|
| Primera vez | Recorre los 33 millones de intentos del zip que ya está en tu Drive para calcular el contexto de red con TODO el tráfico, guarda la muestra y ejecuta | 25 a 40 min |
| Siguientes | Lee la muestra con contexto desde Drive y ejecuta | 10 a 15 min |

Qué cambia respecto a la v4:
- **Señales por IP** (tasa previa de fallos, intentos, usuarios distintos, tiempo desde el intento anterior), **por user agent** (tasa de fallos) y **por cuenta** (fallos previos y recientes, tiempo desde el último intento). Todas usan solo el pasado y ninguna usa etiquetas.
- **El contexto de red se calcula con todo el tráfico del dataset**, como lo vería el sistema en producción, aunque se evalúe sobre el 5 % de usuarios.
- **Se quitan variables que crecen con el tiempo** (volumen acumulado del ASN), que causaban deriva entre validación y prueba.
- **Compara 5 modelos:** reglas, Isolation Forest v4, Isolation Forest v5, supervisado v5 y **híbrido v5** (la propuesta para el consolidador C3).
- Agrega la **tabla de puntos de operación**, que muestra cuántos ataques se detectan según el FPR que se acepte.

In [ ]:
!pip -q install shap

## 1. Código

In [ ]:
%%writefile adaptauth.py
"""
AdaptAuth-ML - pipeline del baseline S8 (Isolation Forest + reglas contextuales).

Implementa el diseño corregido del PI7 (Fase 0):
  - x_i en R^8 (hora cíclica, desviación horaria, historial, frecuencias de país/ASN/dispositivo, RTT)
  - c_i = 0.25 r1 + 0.25 r2 + 0.30 r3 + 0.15 r4 + 0.05 r5
  - s(x) de Isolation Forest (ecuación 1) y reescalado s'(x) (ecuación 1b)
  - R_i = alpha * s'(x_i) + (1 - alpha) * c_i (ecuación 2)
  - Política: acceso directo / OTP / bloqueo, con regla de arranque en frío (P5)
  - Partición temporal 70/15/15 y calibración de alpha, theta1, theta2 solo en validación

Los nombres de columnas siguen el dataset RBA (Wiefling et al., 2022), de modo que el
mismo código corre sobre el archivo real o sobre los datos sintéticos de prueba.
"""
from __future__ import annotations

import json
import time
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import IsolationForest

# --------------------------------------------------------------------------------------
# Columnas del dataset RBA
# --------------------------------------------------------------------------------------
COL_TS = "Login Timestamp"
COL_USER = "User ID"
COL_RTT = "Round-Trip Time [ms]"
COL_IP = "IP Address"
COL_COUNTRY = "Country"
COL_ASN = "ASN"
COL_BROWSER = "Browser Name and Version"
COL_OS = "OS Name and Version"
COL_DEVICE = "Device Type"
COL_SUCCESS = "Login Successful"
COL_ATTACK_IP = "Is Attack IP"          # SOLO etiqueta de evaluación, nunca entrada
COL_ATO = "Is Account Takeover"         # SOLO etiqueta de evaluación, nunca entrada

LABEL_COLUMNS = (COL_ATTACK_IP, COL_ATO)

FEATURES = [
    "hora_sin", "hora_cos", "desv_horaria", "log_n_prev",
    "frec_pais", "frec_asn", "frec_dispositivo", "log_rtt",
]
# Contexto global (v4): no usa etiquetas, solo el pasado de TODA la población
FEATURES_GLOBAL = ["es_movil", "es_desktop", "tasa_fallo_asn", "tasa_fallo_pais", "log_vol_asn"]
# v5: se quitan variables que crecen con el tiempo (log_vol_asn, log_rtt casi sin datos) y se agregan
# señales por IP, por user agent y por fallos de la cuenta
FEATURES_V5 = ["hora_sin", "hora_cos", "desv_horaria", "log_n_prev", "frec_pais", "frec_asn",
               "frec_dispositivo", "es_movil", "es_desktop", "tasa_fallo_asn", "tasa_fallo_pais",
               "tasa_fallo_ua", "ip_tasa_fallo", "ip_log_intentos_prev", "ip_log_usuarios_prev",
               "ip_log_horas_desde_prev", "ip_nueva_para_usuario", "usr_tasa_fallo",
               "usr_fallos_recientes", "usr_log_horas_desde_prev"]
RULES = ["r1_dispositivo_nuevo", "r2_pais_poco_habitual", "r3_viaje_imposible",
         "r4_horario_atipico", "r5_asn_nuevo"]
RULE_WEIGHTS = {"r1_dispositivo_nuevo": 0.25, "r2_pais_poco_habitual": 0.25,
                "r3_viaje_imposible": 0.30, "r4_horario_atipico": 0.15, "r5_asn_nuevo": 0.05}

DECISION_NAMES = {0: "acceso_directo", 1: "otp", 2: "bloqueo"}


@dataclass
class Config:
    seed: int = 42
    n_estimators: int = 100          # t
    max_samples: int = 256           # n
    split: tuple = (0.70, 0.15, 0.15)
    alpha_grid: tuple = (0.3, 0.5, 0.7, 0.85, 1.0)
    theta_grid: tuple = tuple(np.round(np.arange(0.02, 0.951, 0.01), 2))
    max_fpr_theta1: float = 0.05     # RNF02 / CA-C2-02
    max_fpr_theta2: float = 0.01     # bloqueo: tolerancia más estricta
    cold_start_otp: bool = True      # regla P5
    n_test_blocks: int = 10          # bloques temporales para la prueba pareada
    label: str = "anomalo"           # columna de verdad usada para calibrar


# --------------------------------------------------------------------------------------
# 1. Datos
# --------------------------------------------------------------------------------------
def _open_csv(path: str):
    if str(path).endswith(".zip"):          # se lee directo del .zip, sin descomprimir ~9 GB
        import zipfile
        zf = zipfile.ZipFile(path)
        member = next(n for n in zf.namelist() if n.endswith(".csv"))
        return zf.open(member)
    return path


def load_rba(path: str, nrows: int | None = None, user_pct: float | None = None,
             chunksize: int = 2_000_000) -> pd.DataFrame:
    """
    Carga el CSV real de RBA (rba-dataset.csv o el .zip de Zenodo).
      - nrows: primeras N filas (ventana de tiempo corta: muchos usuarios sin historial).
      - user_pct: muestreo POR USUARIO sobre todo el archivo (p. ej. 5 = 5 % de usuarios con
        TODOS sus accesos). Conserva historiales completos; es la opción recomendada.
    """
    usecols = [COL_TS, COL_USER, COL_RTT, COL_IP, COL_COUNTRY, COL_ASN, COL_BROWSER,
               COL_OS, COL_DEVICE, COL_SUCCESS, COL_ATTACK_IP, COL_ATO]
    if str(path).endswith(".parquet"):      # muestra ya filtrada y guardada (lectura en segundos)
        df = pd.read_parquet(path)
        df[COL_TS] = pd.to_datetime(df[COL_TS])
        return df
    if user_pct:
        parts = []
        for ch in pd.read_csv(_open_csv(path), usecols=usecols, chunksize=chunksize):
            h = pd.util.hash_pandas_object(ch[COL_USER], index=False).to_numpy() % 10000
            parts.append(ch[h < user_pct * 100])
        df = pd.concat(parts, ignore_index=True)
    else:
        df = pd.read_csv(_open_csv(path), usecols=usecols, nrows=nrows)
    df[COL_TS] = pd.to_datetime(df[COL_TS])
    for col in (COL_SUCCESS, COL_ATTACK_IP, COL_ATO):
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.lower().isin(["true", "1"])
    return df


CTX_COLS = ["tasa_fallo_asn", "tasa_fallo_pais", "tasa_fallo_ua", "ip_tasa_fallo",
            "ip_log_intentos_prev", "ip_log_usuarios_prev", "ip_log_horas_desde_prev"]


def build_full_context_sample(path: str, user_pct: float, chunksize: int = 2_000_000,
                              m: float = 20.0, verbose: bool = True) -> pd.DataFrame:
    """
    Recorre TODO el archivo en orden temporal y calcula el contexto de red (IP, ASN, país,
    user agent) con el tráfico completo, como lo vería el sistema en producción. Devuelve solo
    los accesos del user_pct % de usuarios, con columnas ctx_* listas para reemplazar las
    calculadas dentro de la muestra. Todo es causal: cada evento solo ve eventos anteriores.
    """
    usecols = [COL_TS, COL_USER, COL_RTT, COL_IP, COL_COUNTRY, COL_ASN, COL_BROWSER,
               COL_OS, COL_DEVICE, COL_SUCCESS, COL_ATTACK_IP, COL_ATO]
    empty = lambda: pd.Series(dtype="float64")
    state = {k: (empty(), empty()) for k in ("asn", "pais", "ua", "ip")}
    ip_users, ip_last = empty(), empty()
    seen = np.empty(0, dtype=np.uint64)
    glob_n = glob_f = 0.0
    t_ref, parts, fila0 = None, [], 0
    t0 = time.time()
    for i, ch in enumerate(pd.read_csv(_open_csv(path), usecols=usecols, chunksize=chunksize)):
        ch = ch.reset_index(drop=True)
        ch["_fila"] = np.arange(fila0, fila0 + len(ch)); fila0 += len(ch)
        ts = pd.to_datetime(ch[COL_TS])
        t_ref = ts.min() if t_ref is None else t_ref
        horas = pd.Series((ts - t_ref).dt.total_seconds().to_numpy() / 3600.0)
        ok = ch[COL_SUCCESS]
        ok = ok.astype(str).str.lower().isin(["true", "1"]) if ok.dtype == object else ok.astype(bool)
        fail = pd.Series((~ok.to_numpy()).astype(int))
        keys = {"asn": ch[COL_ASN].astype(str), "pais": ch[COL_COUNTRY].astype(str),
                "ua": ch[COL_BROWSER].astype(str) + "|" + ch[COL_OS].astype(str) + "|" + ch[COL_DEVICE].astype(str),
                "ip": ch[COL_IP].astype(str)}
        f_cum = fail.cumsum().to_numpy() - fail.to_numpy()
        p0 = (glob_f + f_cum + 1.0) / (glob_n + np.arange(len(ch)) + 4.0)
        prior = {}
        for k, key in keys.items():
            cnt, fl = state[k]
            n_k = key.map(cnt).fillna(0).to_numpy() + fail.groupby(key).cumcount().to_numpy()
            f_k = key.map(fl).fillna(0).to_numpy() + (fail.groupby(key).cumsum() - fail).to_numpy()
            prior[k] = (n_k, f_k)
            state[k] = (cnt.add(key.value_counts(), fill_value=0), fl.add(fail.groupby(key).sum(), fill_value=0))
        out = {}
        for k, name in (("asn", "tasa_fallo_asn"), ("pais", "tasa_fallo_pais"), ("ua", "tasa_fallo_ua")):
            n_k, f_k = prior[k]
            out[name] = (f_k + m * p0) / (n_k + m)
        n_ip, f_ip = prior["ip"]
        out["ip_tasa_fallo"] = (f_ip + 5.0 * out["tasa_fallo_asn"]) / (n_ip + 5.0)
        out["ip_log_intentos_prev"] = np.log1p(n_ip)
        h = pd.util.hash_pandas_object(ch[[COL_IP, COL_USER]].astype(str), index=False).to_numpy()
        first = pd.Series(((~pd.Series(h).duplicated()).to_numpy() & ~np.isin(h, seen)).astype(int))
        users_prev = keys["ip"].map(ip_users).fillna(0).to_numpy() + (first.groupby(keys["ip"]).cumsum() - first).to_numpy()
        out["ip_log_usuarios_prev"] = np.log1p(users_prev)
        ip_users = ip_users.add(first.groupby(keys["ip"]).sum(), fill_value=0)
        seen = np.union1d(seen, h[first.to_numpy() == 1])
        prev = horas.groupby(keys["ip"]).shift(1).fillna(keys["ip"].map(ip_last))
        out["ip_log_horas_desde_prev"] = np.log1p((horas - prev).fillna(24 * 365).clip(lower=0)).to_numpy()
        last = horas.groupby(keys["ip"]).max()
        ip_last = pd.concat([ip_last, last]); ip_last = ip_last[~ip_last.index.duplicated(keep="last")]
        glob_n += len(ch); glob_f += float(fail.sum())
        keep = (pd.util.hash_pandas_object(ch[COL_USER], index=False).to_numpy() % 10000) < user_pct * 100
        sub = ch.loc[keep].copy()
        for name, arr in out.items():
            sub["ctx_" + name] = np.asarray(arr)[keep]
        parts.append(sub)
        if verbose:
            print(f"  bloque {i + 1}: {fila0:,} intentos leídos | IPs vistas {len(ip_users):,} | {time.time() - t0:.0f}s", flush=True)
    df = pd.concat(parts, ignore_index=True)
    df[COL_TS] = pd.to_datetime(df[COL_TS])
    for col in (COL_SUCCESS, COL_ATTACK_IP, COL_ATO):
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.lower().isin(["true", "1"])
    return df


def apply_full_context(d: pd.DataFrame) -> bool:
    """Si la muestra trae columnas ctx_* (tráfico completo), reemplaza las calculadas en la muestra."""
    if not all(("ctx_" + c) in d.columns for c in CTX_COLS):
        return False
    for c in CTX_COLS:
        d[c] = d["ctx_" + c].to_numpy()
    return True


def generate_synthetic_rba(n_users: int = 2000, days: int = 365, seed: int = 42) -> pd.DataFrame:
    """
    Datos SINTÉTICOS con la misma estructura de columnas que RBA. Sirven solo para
    probar el pipeline de punta a punta; sus métricas NO son resultados de la tesis.
    """
    rng = np.random.default_rng(seed)
    start = pd.Timestamp("2020-02-03")
    countries = np.array(["NO", "SE", "DE", "GB", "US", "FR", "ES", "PL", "BR", "IN"])
    home_p = np.array([0.55, 0.08, 0.07, 0.06, 0.06, 0.05, 0.04, 0.04, 0.03, 0.02])
    asn_pool = {c: [10000 + 100 * i + k for k in range(6)] for i, c in enumerate(countries)}
    attacker_countries = np.array(["US", "RU", "CN", "VN", "BR", "NL", "UA", "ID"])
    attacker_asns = [64000 + k for k in range(60)]       # proveedores de hosting
    browsers = ["Chrome 88", "Firefox 85", "Safari 14", "Edge 88", "Chrome Mobile 88",
                "Samsung Internet 13", "Opera 74"]
    oses = ["Windows 10", "Mac OS X 10.15", "iOS 14.4", "Android 10", "Linux", "Android 11"]
    devices = ["desktop", "mobile", "tablet"]
    base_rtt = {c: v for c, v in zip(countries, [30, 45, 60, 65, 140, 70, 90, 70, 220, 260])}

    rows = []
    for uid in range(n_users):
        home = rng.choice(countries, p=home_p)
        user_asns = rng.choice(asn_pool[home], size=rng.integers(1, 3), replace=False)
        user_ips = [f"10.{uid % 250}.{k}.{rng.integers(255)}" for k in range(rng.integers(1, 4))]
        n_dev = rng.integers(1, 4)
        user_devs = [(rng.choice(browsers), rng.choice(oses), rng.choice(devices, p=[.5, .42, .08]))
                     for _ in range(n_dev)]
        pref_hour = np.clip(rng.normal(13, 3), 6, 22)
        n_events = int(np.clip(rng.lognormal(4.2, 0.9), 2, 900))
        times = np.sort(rng.uniform(0, days, n_events))
        for t in times:
            day = np.floor(t)
            hour = (rng.normal(pref_hour, 2.5)) % 24
            ts = start + pd.Timedelta(days=float(day)) + pd.Timedelta(hours=float(hour))
            ctry, asn = home, rng.choice(user_asns)
            if rng.random() < 0.02:                       # viaje legítimo
                ctry = rng.choice(countries)
                asn = rng.choice(asn_pool[ctry])
            elif rng.random() < 0.08:                     # red móvil / otra red del país
                asn = rng.choice(asn_pool[home])
            dev = user_devs[rng.integers(len(user_devs))]
            if rng.random() < 0.03:                       # dispositivo nuevo legítimo
                dev = (rng.choice(browsers), rng.choice(oses), rng.choice(devices))
                user_devs.append(dev)
            rtt = rng.lognormal(np.log(base_rtt[ctry]), 0.35)
            ip = (f"100.64.{rng.integers(40)}.{rng.integers(255)}" if rng.random() < 0.15   # NAT móvil compartida
                  else user_ips[rng.integers(len(user_ips))])
            rows.append((ts, uid, rtt, ctry, asn, dev[0], dev[1], dev[2],
                         rng.random() > 0.04, 0, 0, ip))

    n_legit = len(rows)
    # Relleno de credenciales (credential stuffing) desde IPs de ataque
    n_attack = int(n_legit * 0.08)
    for _ in range(n_attack):
        uid = int(rng.integers(n_users))
        ts = start + pd.Timedelta(days=float(rng.uniform(0, days)))
        ctry = rng.choice(attacker_countries)
        rtt = rng.lognormal(np.log(180), 0.6)
        rows.append((ts, uid, rtt, ctry, rng.choice(attacker_asns), rng.choice(browsers),
                     rng.choice(oses), "desktop", rng.random() < 0.02, 1, 0,
                     f"203.0.{rng.integers(150)}.1"))
    # Toma de cuenta (account takeover): acceso exitoso del atacante
    n_ato = int(n_legit * 0.002)
    for _ in range(n_ato):
        uid = int(rng.integers(n_users))
        ts = start + pd.Timedelta(days=float(rng.uniform(30, days)))
        ctry = rng.choice(attacker_countries)
        rtt = rng.lognormal(np.log(180), 0.6)
        rows.append((ts, uid, rtt, ctry, rng.choice(attacker_asns), rng.choice(browsers),
                     rng.choice(oses), rng.choice(devices), True, 1, 1, f"203.0.{rng.integers(150)}.1"))

    df = pd.DataFrame(rows, columns=[COL_TS, COL_USER, COL_RTT, COL_COUNTRY, COL_ASN,
                                     COL_BROWSER, COL_OS, COL_DEVICE, COL_SUCCESS,
                                     COL_ATTACK_IP, COL_ATO, COL_IP])
    return df


# --------------------------------------------------------------------------------------
# 2. C1 - Ingesta y feature engineering (solo usa el pasado de cada usuario)
# --------------------------------------------------------------------------------------
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Construye x_i (R^8), las reglas r1..r5 y c_i. El historial de cada usuario se forma
    SOLO con sus inicios de sesión exitosos anteriores al evento, de modo que no hay
    fuga temporal ni se usan las etiquetas de ataque como entrada.
    """
    d = df.sort_values(COL_TS, kind="mergesort").reset_index(drop=True).copy()
    d["anomalo"] = ((d[COL_ATTACK_IP] == 1) | (d[COL_ATO] == 1)).astype(int)
    d["dispositivo"] = (d[COL_BROWSER].astype(str) + "|" + d[COL_OS].astype(str) + "|"
                        + d[COL_DEVICE].astype(str))
    succ = d[COL_SUCCESS].astype(bool).astype(int)
    d["_succ"] = succ

    hour = d[COL_TS].dt.hour + d[COL_TS].dt.minute / 60.0
    ang = 2 * np.pi * hour / 24.0
    d["hora_sin"], d["hora_cos"] = np.sin(ang), np.cos(ang)

    g = d.groupby(COL_USER)
    n_prev = g["_succ"].cumsum() - succ                   # accesos exitosos previos
    d["n_prev"] = n_prev
    d["log_n_prev"] = np.log1p(n_prev)

    def prior_share(key: str) -> pd.Series:
        same = d.groupby([COL_USER, key])["_succ"].cumsum() - succ
        return np.where(n_prev > 0, same / n_prev.clip(lower=1), 0.0)

    d["frec_pais"] = prior_share(COL_COUNTRY)
    d["frec_asn"] = prior_share(COL_ASN)
    d["frec_dispositivo"] = prior_share("dispositivo")

    # Hora habitual: media circular de los accesos exitosos previos
    s_cum = (d["hora_sin"] * succ).groupby(d[COL_USER]).cumsum() - d["hora_sin"] * succ
    c_cum = (d["hora_cos"] * succ).groupby(d[COL_USER]).cumsum() - d["hora_cos"] * succ
    mean_ang = np.arctan2(s_cum, c_cum)
    diff = np.abs(np.angle(np.exp(1j * (ang - mean_ang))))     # 0..pi
    desv_h = diff * 24 / (2 * np.pi)                            # 0..12 h
    d["desv_horas"] = np.where(n_prev > 0, desv_h, np.nan)
    d["desv_horaria"] = np.where(n_prev > 0, desv_h / 12.0, 0.5)

    d["log_rtt"] = np.log1p(d[COL_RTT].astype(float).fillna(d[COL_RTT].median()))

    # Último acceso exitoso previo (para viaje imposible)
    d["_ctry_s"] = d[COL_COUNTRY].where(succ == 1)
    d["_ts_s"] = d[COL_TS].where(succ == 1)
    # Vectorizado (sin lambda por usuario) para soportar millones de filas
    d["prev_pais"] = d.groupby(COL_USER)["_ctry_s"].shift(1).groupby(d[COL_USER]).ffill()
    d["prev_ts"] = d.groupby(COL_USER)["_ts_s"].shift(1).groupby(d[COL_USER]).ffill()
    horas_desde_prev = (d[COL_TS] - d["prev_ts"]).dt.total_seconds() / 3600.0

    hist = n_prev >= 1
    d["r1_dispositivo_nuevo"] = (hist & (d["frec_dispositivo"] == 0)).astype(int)
    d["r2_pais_poco_habitual"] = (hist & (d["frec_pais"] < 0.2)).astype(int)
    d["r3_viaje_imposible"] = (d["prev_pais"].notna() & (d["prev_pais"] != d[COL_COUNTRY])
                               & (horas_desde_prev < 2)).astype(int)
    d["r4_horario_atipico"] = ((n_prev >= 5) & (d["desv_horas"] > 4)).astype(int)
    d["r5_asn_nuevo"] = (hist & (d["frec_asn"] == 0)).astype(int)
    d["c_i"] = sum(RULE_WEIGHTS[r] * d[r] for r in RULES)
    d["arranque_frio"] = (n_prev == 0).astype(int)

    # ---- Contexto global (v4), causal: cada evento solo ve eventos anteriores ----
    dev = d[COL_DEVICE].astype(str).str.lower()
    d["es_movil"] = (dev == "mobile").astype(int)
    d["es_desktop"] = (dev == "desktop").astype(int)
    fail = 1 - succ
    g_n = np.arange(len(d))
    g_f = fail.cumsum().to_numpy() - fail.to_numpy()
    p0 = (g_f + 1.0) / (g_n + 4.0)                      # tasa global de fallo previa (suavizada)
    m = 20.0                                            # fuerza del suavizado bayesiano
    for key, name in ((COL_ASN, "tasa_fallo_asn"), (COL_COUNTRY, "tasa_fallo_pais")):
        n_k = d.groupby(key).cumcount().to_numpy()
        f_k = fail.groupby(d[key]).cumsum().to_numpy() - fail.to_numpy()
        d[name] = (f_k + m * p0) / (n_k + m)
    d["log_vol_asn"] = np.log1p(d.groupby(COL_ASN).cumcount())

    # ---- v5: IP, user agent y fallos de la cuenta (causales, sin etiquetas) ----
    horas = (d[COL_TS] - d[COL_TS].min()).dt.total_seconds() / 3600.0
    ip = d[COL_IP].astype(str)
    ip_n = d.groupby(ip).cumcount()
    ip_f = fail.groupby(ip).cumsum() - fail
    d["ip_tasa_fallo"] = (ip_f + 5.0 * d["tasa_fallo_asn"]) / (ip_n + 5.0)   # previa jerárquica: ASN
    d["ip_log_intentos_prev"] = np.log1p(ip_n)
    first_pair = (~d.duplicated([COL_IP, COL_USER])).astype(int)
    d["ip_log_usuarios_prev"] = np.log1p(first_pair.groupby(ip).cumsum() - first_pair)
    d["ip_log_horas_desde_prev"] = np.log1p(horas.groupby(ip).diff().fillna(24 * 365).clip(lower=0))
    pair_ok_prev = d.groupby([COL_USER, COL_IP])["_succ"].cumsum() - succ
    d["ip_nueva_para_usuario"] = (pair_ok_prev == 0).astype(int)
    ua = d["dispositivo"]
    ua_n = d.groupby(ua).cumcount()
    ua_f = fail.groupby(ua).cumsum() - fail
    d["tasa_fallo_ua"] = (ua_f + m * p0) / (ua_n + m)
    u_n = d.groupby(COL_USER).cumcount()
    u_f = fail.groupby(d[COL_USER]).cumsum() - fail                     # fallos previos de la cuenta
    d["usr_tasa_fallo"] = (u_f + 3.0 * p0) / (u_n + 3.0)
    d["usr_fallos_recientes"] = u_f - u_f.groupby(d[COL_USER]).shift(5).fillna(0)   # en sus 5 intentos previos
    d["usr_log_horas_desde_prev"] = np.log1p(horas.groupby(d[COL_USER]).diff().fillna(24 * 365).clip(lower=0))

    return d.drop(columns=["_succ", "_ctry_s", "_ts_s"])


def diagnose(raw: pd.DataFrame, d: pd.DataFrame) -> dict:
    """
    Análisis exploratorio orientado a la detección: ¿qué variables separan de verdad los
    accesos anómalos de los legítimos? (Etapa 1 del procedimiento metodológico).
    """
    from sklearn.metrics import roc_auc_score
    y = d["anomalo"].to_numpy()
    lab = np.where(y == 1, "anomalo", "legitimo")
    per_user = d.groupby(COL_USER).size()

    def auc(col, mask=None):
        m = np.ones(len(d), bool) if mask is None else mask
        yy, xx = y[m], d.loc[m, col].astype(float).fillna(0).to_numpy()
        return round(float(roc_auc_score(yy, xx)), 3) if 0 < yy.sum() < len(yy) else None

    hist = (d["n_prev"] >= 1).to_numpy()
    cols = FEATURES + FEATURES_GLOBAL + RULES + ["c_i", COL_RTT]

    def top_share(col, k=5):
        out = {}
        for name in ("legitimo", "anomalo"):
            s = d.loc[lab == name, col].astype(str).value_counts(normalize=True).head(k)
            out[name] = {i: round(float(v), 3) for i, v in s.items()}
        return out

    legit_asn = set(d.loc[y == 0, COL_ASN])
    legit_users = set(d.loc[(y == 0) & d[COL_SUCCESS].astype(bool), COL_USER])
    atk = d[y == 1]
    return {
        "archivo_ordenado_por_fecha": bool(raw[COL_TS].is_monotonic_increasing),
        "rango_fechas": [str(d[COL_TS].min()), str(d[COL_TS].max())],
        "eventos": int(len(d)), "usuarios": int(per_user.size),
        "eventos_por_usuario_p50_p90_max": [int(per_user.quantile(.5)), int(per_user.quantile(.9)),
                                            int(per_user.max())],
        "tasa_anomalos": round(float(y.mean()), 4), "n_ato": int(d[COL_ATO].sum()),
        "exito_login": {k: round(float(d.loc[lab == k, COL_SUCCESS].astype(bool).mean()), 4)
                        for k in ("legitimo", "anomalo")},
        "primer_acceso": {k: round(float((d.loc[lab == k, "n_prev"] == 0).mean()), 4)
                          for k in ("legitimo", "anomalo")},
        "anomalos_sobre_usuarios_con_historial_legitimo": round(float(atk[COL_USER].isin(legit_users).mean()), 4),
        "anomalos_con_asn_visto_en_legitimos": round(float(atk[COL_ASN].isin(legit_asn).mean()), 4),
        "pais_top5": top_share(COL_COUNTRY), "tipo_dispositivo_top5": top_share(COL_DEVICE),
        "asn_top5": top_share(COL_ASN),
        "auc_univariada_todos": {c: auc(c) for c in cols},
        "auc_univariada_con_historial": {c: auc(c, hist) for c in cols},
        "tasa_reglas": {r: {k: round(float(d.loc[lab == k, r].mean()), 4) for k in ("legitimo", "anomalo")}
                        for r in RULES},
        "nota_auc": "0.5 = no separa; > 0.5 = valores altos indican anomalía; < 0.5 = relación invertida",
    }


def temporal_split(d: pd.DataFrame, split=(0.70, 0.15, 0.15)):
    n = len(d)
    a, b = int(n * split[0]), int(n * (split[0] + split[1]))
    return d.iloc[:a].copy(), d.iloc[a:b].copy(), d.iloc[b:].copy()


# --------------------------------------------------------------------------------------
# 3. C2 / C3 / C4 - detección, consolidación y decisión
# --------------------------------------------------------------------------------------
def train_iforest(train: pd.DataFrame, cfg: Config, features=None) -> IsolationForest:
    features = features or FEATURES
    model = IsolationForest(n_estimators=cfg.n_estimators, max_samples=cfg.max_samples,
                            contamination="auto", random_state=cfg.seed, n_jobs=-1)
    model.fit(train[features].to_numpy())
    return model


def anomaly_score(model: IsolationForest, d: pd.DataFrame, features=None) -> np.ndarray:
    """Ecuación 1: s(x, n) = 2^(-E[h(x)]/c(n)). En scikit-learn, score_samples = -s(x, n)."""
    return -model.score_samples(d[features or FEATURES].to_numpy())


def rescale(s: np.ndarray, s_ref: float) -> np.ndarray:
    """Ecuación 1b: s'(x) = max(0, (s - s_ref) / (1 - s_ref))."""
    return np.clip((s - s_ref) / (1.0 - s_ref), 0.0, 1.0)


def risk_score(s_prime: np.ndarray, c: np.ndarray, alpha: float) -> np.ndarray:
    """Ecuación 2: R_i = alpha * s'(x_i) + (1 - alpha) * c_i."""
    return alpha * s_prime + (1.0 - alpha) * c


def decide(R: np.ndarray, theta1: float, theta2: float, cold: np.ndarray | None = None,
           cold_start_otp: bool = True) -> np.ndarray:
    """Reglas P2, P3 y P5. 0 = acceso directo, 1 = OTP, 2 = bloqueo."""
    dec = np.where(R > theta2, 2, np.where(R > theta1, 1, 0))
    if cold_start_otp and cold is not None:
        dec = np.where((cold == 1) & (dec == 0), 1, dec)
    return dec


# --------------------------------------------------------------------------------------
# 4. Métricas (dos análisis binarios sobre la decisión de tres niveles)
# --------------------------------------------------------------------------------------
def binary_metrics(y: np.ndarray, pred: np.ndarray) -> dict:
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    fpr = fp / (fp + tn) if fp + tn else 0.0
    fnr = fn / (fn + tp) if fn + tp else 0.0
    denom = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn) - (fp * fn)) / denom if denom else 0.0
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, precision=precision, recall=recall, f1=f1,
                fpr=fpr, fnr=fnr, mcc=mcc)


def evaluate_decisions(y: np.ndarray, dec: np.ndarray) -> dict:
    """Análisis theta1: positivo = OTP o bloqueo. Análisis theta2: positivo = bloqueo."""
    return {"analisis_theta1": binary_metrics(y, (dec >= 1).astype(int)),
            "analisis_theta2": binary_metrics(y, (dec == 2).astype(int))}


def confusion_3x2(y: np.ndarray, dec: np.ndarray) -> pd.DataFrame:
    tab = pd.crosstab(pd.Series(dec).map(DECISION_NAMES).rename("decision"),
                      pd.Series(y).map({0: "legitimo", 1: "anomalo"}).rename("etiqueta_real"))
    return tab.reindex(index=list(DECISION_NAMES.values()), columns=["legitimo", "anomalo"],
                       fill_value=0)


# --------------------------------------------------------------------------------------
# 5. Calibración en validación (el conjunto de prueba no se usa aquí)
# --------------------------------------------------------------------------------------
def calibrate(val: pd.DataFrame, s_prime: np.ndarray, cfg: Config, alpha_grid) -> dict:
    """
    Búsqueda en malla: maximiza F1 del análisis theta1 sujeto a FPR <= max_fpr_theta1.
    theta2 es el menor umbral > theta1 cuyo FPR de bloqueo <= max_fpr_theta2.
    """
    y = val[cfg.label].to_numpy()
    cold = val["arranque_frio"].to_numpy()
    c = val["c_i"].to_numpy()
    best, fallback = None, None
    for alpha in alpha_grid:
        R = risk_score(s_prime, c, alpha)
        for t1 in cfg.theta_grid:
            dec = decide(R, t1, 1.01, cold, cfg.cold_start_otp)
            m = binary_metrics(y, (dec >= 1).astype(int))
            cand = dict(alpha=float(alpha), theta1=float(t1), f1_val=m["f1"], fpr_val=m["fpr"])
            if m["fpr"] <= cfg.max_fpr_theta1 and (best is None or m["f1"] > best["f1_val"]):
                best = cand
            # respaldo: menor FPR posible y, a igualdad, mayor F1
            if fallback is None or (m["fpr"], -m["f1"]) < (fallback["fpr_val"], -fallback["f1_val"]):
                fallback = cand
    if best is None:   # ninguna combinación cumple FPR <= umbral: se declara y se usa el respaldo
        best = dict(fallback, restriccion_fpr_incumplida=True)
    R = risk_score(s_prime, c, best["alpha"])
    best["theta2"] = 0.95
    for t2 in cfg.theta_grid:
        if t2 <= best["theta1"]:
            continue
        m = binary_metrics(y, (R > t2).astype(int))
        if m["fpr"] <= cfg.max_fpr_theta2:
            best["theta2"] = float(t2)
            break
    return best


# --------------------------------------------------------------------------------------
# 6. Prueba pareada por bloques temporales (baseline vs AdaptAuth-ML)
# --------------------------------------------------------------------------------------
def paired_block_test(test: pd.DataFrame, dec_a: np.ndarray, dec_b: np.ndarray,
                      y_col: str, n_blocks: int) -> dict:
    """
    Divide el conjunto de prueba en n bloques temporales consecutivos y compara el F1
    (análisis theta1) de ambas condiciones sobre los MISMOS bloques: muestras pareadas.
    """
    y = test[y_col].to_numpy()
    idx = np.array_split(np.arange(len(test)), n_blocks)
    f1_a = np.array([binary_metrics(y[i], (dec_a[i] >= 1).astype(int))["f1"] for i in idx])
    f1_b = np.array([binary_metrics(y[i], (dec_b[i] >= 1).astype(int))["f1"] for i in idx])
    diff = f1_b - f1_a
    shapiro_p = float(stats.shapiro(diff).pvalue) if np.ptp(diff) > 0 else 1.0
    if shapiro_p > 0.05:
        t = stats.ttest_rel(f1_b, f1_a)
        test_name, stat, p = "t de Student pareada", float(t.statistic), float(t.pvalue)
    else:
        w = stats.wilcoxon(f1_b, f1_a)
        test_name, stat, p = "Wilcoxon de rangos con signo", float(w.statistic), float(w.pvalue)
    sd = diff.std(ddof=1)
    dz = float(diff.mean() / sd) if sd > 0 else float("inf")
    return dict(n_bloques=n_blocks, f1_baseline=f1_a.tolist(), f1_adaptauth=f1_b.tolist(),
                media_baseline=float(f1_a.mean()), media_adaptauth=float(f1_b.mean()),
                de_baseline=float(f1_a.std(ddof=1)), de_adaptauth=float(f1_b.std(ddof=1)),
                shapiro_p_diferencias=shapiro_p, prueba=test_name, estadistico=stat, p_valor=p,
                d_cohen_dz=dz)


# --------------------------------------------------------------------------------------
# 7. Latencia (RNF03 / CA-C2-03)
# --------------------------------------------------------------------------------------
def inference_latency_ms(model, d: pd.DataFrame, n: int = 300, seed: int = 0, features=None) -> dict:
    rng = np.random.default_rng(seed)
    X = d[features or FEATURES].to_numpy()
    score = model.score_samples if hasattr(model, "score_samples") else model.predict_proba
    times = []
    for i in rng.integers(0, len(X), n):
        t0 = time.perf_counter()
        score(X[i:i + 1])
        times.append((time.perf_counter() - t0) * 1000)
    t = np.array(times)
    return dict(p50_ms=float(np.percentile(t, 50)), p95_ms=float(np.percentile(t, 95)),
                p99_ms=float(np.percentile(t, 99)), media_ms=float(t.mean()))


def to_json(obj, path):
    def conv(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, Config):
            return asdict(o)
        raise TypeError(type(o))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=conv)


In [ ]:
%%writefile figures.py
"""Figuras del experimento base (paleta de referencia: azul y naranja, marcas finas)."""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11, "axes.edgecolor": GRID,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150,
})


def _grid(ax, axis="y"):
    ax.grid(axis=axis, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def risk_distribution(R, y, t1, t2, path):
    fig, ax = plt.subplots(figsize=(8, 4.2))
    bins = np.linspace(0, 1, 41)
    ax.hist(R[y == 0], bins=bins, density=True, histtype="step", linewidth=2, color=BLUE,
            label="Accesos legítimos")
    ax.hist(R[y == 1], bins=bins, density=True, histtype="step", linewidth=2, color=ORANGE,
            label="Accesos anómalos")
    ax.set_yscale("log")
    top = ax.get_ylim()[1]
    for t, name, f, ha, dx in ((t1, "θ1", 0.5, "right", -0.01), (t2, "θ2", 0.12, "left", 0.01)):
        ax.axvline(t, color=INK2, linewidth=1, linestyle="--")
        ax.text(t + dx, top * f, f"{name} = {t:.2f}", color=INK2, fontsize=10, ha=ha)
    ax.set_xlabel("Score de riesgo consolidado R_i")
    ax.set_ylabel("Densidad (escala log)")
    _grid(ax)
    ax.legend(frameon=False, loc="upper right")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def paired_blocks(res, path, etiquetas=("Baseline (solo reglas)", "AdaptAuth-ML")):
    a, b = np.array(res["f1_baseline"]), np.array(res["f1_adaptauth"])
    x = np.arange(1, len(a) + 1)
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.plot(x, a, color=BLUE, linewidth=2, marker="o", markersize=7, label=etiquetas[0])
    ax.plot(x, b, color=ORANGE, linewidth=2, marker="o", markersize=7, label=etiquetas[1])
    ax.set_xticks(x)
    ax.set_xlabel("Bloque temporal del conjunto de prueba")
    ax.set_ylabel("F1 (análisis θ1)")
    ax.set_ylim(0, 1)
    _grid(ax)
    ax.legend(frameon=False, loc="lower right")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def shap_bars(mean_abs, path, xlabel="Importancia media |SHAP| sobre el score de anomalía"):
    s = mean_abs.sort_values()
    fig, ax = plt.subplots(figsize=(8, 0.32 * len(s) + 1.4))
    ax.barh(s.index, s.values, color=BLUE, height=0.6)
    for i, v in enumerate(s.values):
        ax.text(v, i, f" {v:.3f}", va="center", color=INK2, fontsize=9)
    ax.set_xlabel(xlabel)
    _grid(ax, "x")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def model_comparison(tabla, path):
    """Barras horizontales agrupadas: AUC-ROC y F1 (θ1) por modelo, misma escala 0 a 1."""
    t = tabla.iloc[::-1].reset_index(drop=True)
    y = np.arange(len(t))
    h = 0.36
    fig, ax = plt.subplots(figsize=(8.5, 0.6 * len(t) + 1.6))
    ax.barh(y + h / 2 + 0.01, t["auc_roc"], height=h, color=BLUE, label="AUC-ROC")
    ax.barh(y - h / 2 - 0.01, t["f1_theta1"], height=h, color=ORANGE, label="F1 (análisis θ1)")
    for i, (a, f) in enumerate(zip(t["auc_roc"], t["f1_theta1"])):
        ax.text(a + 0.01, i + h / 2, f"{a:.2f}", va="center", color=INK2, fontsize=9)
        ax.text(f + 0.01, i - h / 2, f"{f:.2f}", va="center", color=INK2, fontsize=9)
    ax.axvline(0.5, color=INK2, linewidth=1, linestyle="--")
    ax.text(0.505, -0.75, "azar (AUC 0.5)", color=INK2, fontsize=9)
    ax.set_yticks(y)
    ax.set_yticklabels(t["modelo"].str.replace("_", " "))
    ax.set_xlim(0, 1.08)
    ax.set_ylim(-0.9, len(t) - 0.4)
    _grid(ax, "x")
    ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def operating_points(po, path, titulo=""):
    """Recall y precisión en prueba según el FPR fijado en validación (puntos de operación)."""
    fig, ax = plt.subplots(figsize=(8, 4.2))
    x = po["fpr_prueba"] * 100
    ax.plot(x, po["recall"], color=BLUE, linewidth=2, marker="o", markersize=7, label="Recall (ataques detectados)")
    ax.plot(x, po["precision"], color=ORANGE, linewidth=2, marker="o", markersize=7, label="Precisión")
    for xi, f in zip(x, po["f1"]):
        ax.annotate(f"F1 {f:.2f}", (xi, 0.02), color=INK2, fontsize=8, ha="center")
    ax.set_xscale("log")
    ax.set_xticks(x); ax.set_xticklabels([f"{v:.1f}%" for v in x], fontsize=9)
    ax.minorticks_off()
    ax.set_xlabel("FPR en prueba (legítimos a los que se les pide OTP)")
    ax.set_ylim(0, 1.02)
    _grid(ax)
    ax.legend(frameon=False, loc="center right")
    if titulo:
        ax.set_title(f"Puntos de operación: {titulo.replace('_', ' ')}", color=INK2, fontsize=11, loc="left")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


In [ ]:
%%writefile run_experimento_v5.py
"""
Experimento v5 de AdaptAuth-ML: objetivo F1 >= 0.60 en el análisis θ1 (OTP o bloqueo).

Diagnóstico que lo motiva (v4, RBA real):
  - Con 7.6 % de ataques y FPR <= 5 %, F1 = 0.60 exige recall >= 0.5-0.7; v4 lograba 0.15.
  - Las variables de grano grueso (país, ASN) no distinguen legítimos y ataques que comparten la
    misma red. La etiqueta del RBA es por IP (Is Attack IP), así que se agregan señales por IP,
    por user agent y por fallos de la cuenta, todas causales y sin etiquetas.
  - Se quitan variables que crecen con el tiempo (deriva entre validación y prueba).

Modelos (calibrados en validación, evaluados en prueba):
    reglas_estaticas    baseline Alternativa 1 (c_i)
    iforest_v4          diseño v4 (continuidad)
    iforest_v5          Isolation Forest con FEATURES_V5 (no supervisado)
    supervisado_v5      HistGradientBoosting con FEATURES_V5 (aprende patrones de ataque conocidos)
    hibrido_v5          R = alpha * s'(IF v5) + (1 - alpha) * p(supervisado)   <- propuesta C3

Uso:
    python run_experimento_v5.py --data ruta/rba_muestra_usuarios_5pct.parquet
    python run_experimento_v5.py --synthetic
"""
import argparse
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score

sys.path.insert(0, str(Path(__file__).parent / "src"))
import adaptauth as aa  # noqa: E402
import figures  # noqa: E402

OUT = Path(__file__).parent / "results"
F_V4 = aa.FEATURES + aa.FEATURES_GLOBAL
F_V5 = aa.FEATURES_V5
FPR_OBJETIVOS = (0.005, 0.01, 0.02, 0.03, 0.05, 0.10, 0.20)


def f1_max(y, score):
    p, r, _ = precision_recall_curve(y, score)
    f = np.where(p + r > 0, 2 * p * r / np.maximum(p + r, 1e-12), 0)
    return float(f.max())


def evaluar(nombre, s_val, s_test, val, test, cfg, alpha_grid=(1.0,), c_val=None, c_test=None):
    v = val.assign(c_i=0.0 if c_val is None else c_val)
    cal = aa.calibrate(v, s_val, cfg, alpha_grid)
    c = np.zeros(len(test)) if c_test is None else c_test
    R = aa.risk_score(s_test, c, cal["alpha"])
    R_val = aa.risk_score(s_val, v["c_i"].to_numpy(), cal["alpha"])
    dec = aa.decide(R, cal["theta1"], cal["theta2"], test["arranque_frio"].to_numpy(), cfg.cold_start_otp)
    y, yv = test[cfg.label].to_numpy(), val[cfg.label].to_numpy()
    res = dict(calibracion=cal, **aa.evaluate_decisions(y, dec),
               auc_roc=float(roc_auc_score(y, R)), auc_pr=float(average_precision_score(y, R)),
               f1_max_validacion=f1_max(yv, R_val),
               f1_max_oraculo_prueba=f1_max(y, R),   # solo diagnóstico: usa etiquetas de prueba
               score_medio_legitimos=float(R[y == 0].mean()))
    m = res["analisis_theta1"]
    print(f"  {nombre:<18} AUC-ROC={res['auc_roc']:.3f} AUC-PR={res['auc_pr']:.3f} | θ1: F1={m['f1']:.3f} "
          f"P={m['precision']:.3f} R={m['recall']:.3f} FPR={m['fpr']:.3f} | F1 máx val={res['f1_max_validacion']:.3f}"
          f" oráculo={res['f1_max_oraculo_prueba']:.3f}" + ("  [FPR<=5% no alcanzable]" if cal.get("restriccion_fpr_incumplida") else ""))
    return res, dec, R, R_val


def puntos_operacion(R_val, R_test, val, test, label):
    """Umbral fijado en validación para cada FPR objetivo; métricas reales en prueba."""
    yv, yt = val[label].to_numpy(), test[label].to_numpy()
    filas = []
    for fpr in FPR_OBJETIVOS:
        thr = float(np.quantile(R_val[yv == 0], 1 - fpr))
        m = aa.binary_metrics(yt, (R_test > thr).astype(int))
        filas.append(dict(fpr_objetivo=fpr, umbral=thr, fpr_prueba=m["fpr"], recall=m["recall"],
                          precision=m["precision"], f1=m["f1"]))
    return pd.DataFrame(filas)


def diagnostico_etiquetas(d, poblacion):
    """¿Qué tan 'aprendible' es la etiqueta Is Attack IP? (naturaleza por IP, mezcla con legítimos)."""
    from sklearn.metrics import roc_auc_score as auc
    g = d.groupby(aa.COL_IP)
    por_ip = pd.DataFrame({"ataque": g[aa.COL_ATTACK_IP].mean(), "usuarios": g[aa.COL_USER].nunique(),
                           "intentos": g.size(), "fallo": 1 - g[aa.COL_SUCCESS].mean()})
    atk_ip, leg_ip = por_ip[por_ip.ataque > 0], por_ip[por_ip.ataque == 0]
    ok = poblacion[poblacion["anomalo"] == 1]
    usuarios_con_ip_limpia = set(poblacion.loc[poblacion["anomalo"] == 0, aa.COL_USER])
    y = poblacion["anomalo"].to_numpy()
    return {
        "ips_con_etiqueta_mixta": round(float(((por_ip.ataque > 0) & (por_ip.ataque < 1)).mean()), 4),
        "ips_ataque": int(len(atk_ip)), "ips_legitimas": int(len(leg_ip)),
        "usuarios_por_ip_ataque_p50_p90_max": [float(atk_ip.usuarios.quantile(q)) for q in (.5, .9)] + [int(atk_ip.usuarios.max())],
        "usuarios_por_ip_legitima_p50_p90_max": [float(leg_ip.usuarios.quantile(q)) for q in (.5, .9)] + [int(leg_ip.usuarios.max())],
        "tasa_fallo_media_ip_ataque": round(float(atk_ip.fallo.mean()), 4),
        "tasa_fallo_media_ip_legitima": round(float(leg_ip.fallo.mean()), 4),
        "exitos_desde_ip_ataque_de_usuarios_que_tambien_entran_desde_ip_limpia":
            round(float(ok[aa.COL_USER].isin(usuarios_con_ip_limpia).mean()), 4),
        "auc_univariada_v5_poblacion": {c: round(float(auc(y, poblacion[c].astype(float))), 3) for c in F_V5},
        "nota": "AUC 0.5 = no separa; > 0.5 valores altos indican ataque; < 0.5 relación invertida",
    }


def main():
    ap = argparse.ArgumentParser()
    g = ap.add_mutually_exclusive_group(required=True)
    g.add_argument("--data")
    g.add_argument("--synthetic", action="store_true")
    ap.add_argument("--no-shap", action="store_true")
    args = ap.parse_args()

    cfg = aa.Config(cold_start_otp=False)
    OUT.mkdir(exist_ok=True); (OUT / "figuras").mkdir(exist_ok=True)
    t0 = time.time()
    raw = aa.generate_synthetic_rba(seed=cfg.seed) if args.synthetic else aa.load_rba(args.data)
    d = aa.build_features(raw)
    contexto_completo = aa.apply_full_context(d)
    print("[contexto] IP/ASN/país/UA calculados con " + ("TODO el tráfico del dataset" if contexto_completo
          else "el tráfico de la muestra (más débil)"))
    pob = d[d[aa.COL_SUCCESS].astype(bool)].reset_index(drop=True)
    train, val, test = aa.temporal_split(pob, cfg.split)
    diag = diagnostico_etiquetas(d, pob)
    aa.to_json(diag, OUT / "diagnostico_etiquetas.json")
    print(f"[datos] {len(d):,} intentos | {len(pob):,} con contraseña correcta | prueba: {len(test):,} "
          f"({test['anomalo'].mean():.1%} ataques) | {time.time() - t0:.0f}s")
    print(f"[etiqueta] IPs mixtas={diag['ips_con_etiqueta_mixta']} | usuarios/IP ataque p50,p90={diag['usuarios_por_ip_ataque_p50_p90_max'][:2]}"
          f" vs legítima {diag['usuarios_por_ip_legitima_p50_p90_max'][:2]} | fallo IP ataque={diag['tasa_fallo_media_ip_ataque']}"
          f" vs legítima {diag['tasa_fallo_media_ip_legitima']}")

    yv_leg = val[cfg.label].to_numpy() == 0
    res, decs, Rs, Rvs = {}, {}, {}, {}
    print("[modelos]")
    res["reglas_estaticas"], decs["reglas_estaticas"], Rs["reglas_estaticas"], _ = evaluar(
        "reglas_estaticas", np.zeros(len(val)), np.zeros(len(test)), val, test, cfg, (0.0,),
        val["c_i"].to_numpy(), test["c_i"].to_numpy())

    ifs = {}
    for nombre, feats in (("iforest_v4", F_V4), ("iforest_v5", F_V5)):
        mdl = aa.train_iforest(train, cfg, feats)
        s_v, s_t = aa.anomaly_score(mdl, val, feats), aa.anomaly_score(mdl, test, feats)
        s_ref = float(np.median(s_v[yv_leg]))
        sp_v, sp_t = aa.rescale(s_v, s_ref), aa.rescale(s_t, s_ref)
        res[nombre], decs[nombre], Rs[nombre], Rvs[nombre] = evaluar(nombre, sp_v, sp_t, val, test, cfg)
        ifs[nombre] = (mdl, sp_v, sp_t)

    hgb = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, max_leaf_nodes=63,
                                         l2_regularization=1.0, early_stopping=True,
                                         validation_fraction=0.1, random_state=cfg.seed)
    hgb.fit(train[F_V5].to_numpy(), train[cfg.label].to_numpy())
    p_v = hgb.predict_proba(val[F_V5].to_numpy())[:, 1]
    p_t = hgb.predict_proba(test[F_V5].to_numpy())[:, 1]
    res["supervisado_v5"], decs["supervisado_v5"], Rs["supervisado_v5"], Rvs["supervisado_v5"] = evaluar(
        "supervisado_v5", p_v, p_t, val, test, cfg)

    # Híbrido (C3): alpha * s'(IF) + (1 - alpha) * p(supervisado); alpha calibrado en validación
    _, sp_v, sp_t = ifs["iforest_v5"]
    res["hibrido_v5"], decs["hibrido_v5"], Rs["hibrido_v5"], Rvs["hibrido_v5"] = evaluar(
        "hibrido_v5", sp_v, sp_t, val, test, cfg, (0.0, 0.1, 0.25, 0.5, 0.75, 1.0), p_v, p_t)

    tabla = pd.DataFrame([dict(
        modelo=k, auc_roc=r["auc_roc"], auc_pr=r["auc_pr"], f1_theta1=r["analisis_theta1"]["f1"],
        precision_theta1=r["analisis_theta1"]["precision"], recall_theta1=r["analisis_theta1"]["recall"],
        fpr_theta1=r["analisis_theta1"]["fpr"], f1_theta2=r["analisis_theta2"]["f1"],
        fpr_theta2=r["analisis_theta2"]["fpr"], mcc_theta1=r["analisis_theta1"]["mcc"],
        f1_max_validacion=r["f1_max_validacion"], f1_max_oraculo_prueba=r["f1_max_oraculo_prueba"],
        alpha=r["calibracion"]["alpha"], theta1=r["calibracion"]["theta1"], theta2=r["calibracion"]["theta2"],
        fpr_5pct_alcanzable=not r["calibracion"].get("restriccion_fpr_incumplida", False))
        for k, r in res.items()])
    tabla.to_csv(OUT / "tabla_comparativa_v5.csv", index=False)

    mejor = max(("iforest_v5", "supervisado_v5", "hibrido_v5"), key=lambda k: res[k]["f1_max_validacion"])
    po = puntos_operacion(Rvs[mejor], Rs[mejor], val, test, cfg.label)
    po.to_csv(OUT / "puntos_operacion_v5.csv", index=False)

    y = test[cfg.label].to_numpy()
    lat_if = aa.inference_latency_ms(ifs["iforest_v5"][0], test, features=F_V5)
    lat_sup = aa.inference_latency_ms(hgb, test, features=F_V5)
    resultados = dict(
        origen_datos="SINTETICO (solo prueba)" if args.synthetic else f"RBA real: {args.data}",
        poblacion="inicios con contraseña correcta",
        contexto_red="trafico completo" if contexto_completo else "solo muestra",
        volumen=dict(intentos=len(d), poblacion=len(pob), train=len(train), val=len(val), test=len(test),
                     ataques_test=int(y.sum()), ato_test=int(test[aa.COL_ATO].sum())),
        modelos=res, mejor_por_validacion=mejor,
        puntos_operacion=po.round(4).to_dict(orient="records"),
        prueba_pareada_hibrido_vs_reglas=aa.paired_block_test(test, decs["reglas_estaticas"], decs["hibrido_v5"], cfg.label, cfg.n_test_blocks),
        prueba_pareada_hibrido_vs_iforest_v4=aa.paired_block_test(test, decs["iforest_v4"], decs["hibrido_v5"], cfg.label, cfg.n_test_blocks),
        prueba_pareada_hibrido_vs_iforest_v5=aa.paired_block_test(test, decs["iforest_v5"], decs["hibrido_v5"], cfg.label, cfg.n_test_blocks),
        latencia_ms=dict(iforest_v5=lat_if, supervisado_v5=lat_sup,
                         hibrido_p99_aprox=lat_if["p99_ms"] + lat_sup["p99_ms"]),
    )
    aa.confusion_3x2(y, decs["hibrido_v5"]).to_csv(OUT / "matriz_3x2_hibrido_v5.csv")

    rng = np.random.default_rng(cfg.seed)
    sel = rng.choice(len(test), size=min(30000, len(test)), replace=False)
    pi = permutation_importance(hgb, test.iloc[sel][F_V5].to_numpy(), y[sel], scoring="average_precision",
                                n_repeats=3, random_state=cfg.seed)
    imp_sup = pd.Series(pi.importances_mean, index=F_V5).sort_values(ascending=False)
    resultados["importancia_supervisado_v5"] = imp_sup.round(4).to_dict()
    figures.shap_bars(imp_sup.clip(lower=0), OUT / "figuras" / "importancia_supervisado_v5.png",
                      xlabel="Caída de AUC-PR al permutar la variable")
    if not args.no_shap:
        import shap
        ex = shap.TreeExplainer(ifs["iforest_v5"][0])
        imp = []
        for b in np.array_split(np.arange(len(test)), 5):
            s = rng.choice(b, size=min(400, len(b)), replace=False)
            imp.append(np.abs(ex.shap_values(test.iloc[s][F_V5].to_numpy())).mean(axis=0))
        imp = pd.DataFrame(imp, columns=F_V5)
        top = set(imp.mean().sort_values(ascending=False).index[:3])
        coinc = sum(set(r.sort_values(ascending=False).index[:3]) == top for _, r in imp.iterrows())
        resultados["shap_iforest_v5"] = dict(top3=list(imp.mean().sort_values(ascending=False).index[:3]),
                                             bloques_que_coinciden=coinc, cumple=coinc >= 4)
        figures.shap_bars(imp.mean(), OUT / "figuras" / "shap_iforest_v5.png")

    c = res["hibrido_v5"]["calibracion"]
    figures.risk_distribution(Rs["hibrido_v5"], y, c["theta1"], c["theta2"], OUT / "figuras" / "distribucion_riesgo_v5.png")
    figures.paired_blocks(resultados["prueba_pareada_hibrido_vs_iforest_v4"], OUT / "figuras" / "f1_por_bloque_v5.png",
                          etiquetas=("AdaptAuth-ML v4", "AdaptAuth-ML v5 (híbrido)"))
    figures.model_comparison(tabla, OUT / "figuras" / "comparacion_modelos_v5.png")
    figures.operating_points(po, OUT / "figuras" / "puntos_operacion_v5.png", titulo=mejor)
    aa.to_json(resultados, OUT / "metricas_v5.json")
    print(tabla[["modelo", "auc_roc", "auc_pr", "f1_theta1", "recall_theta1", "fpr_theta1",
                 "f1_max_validacion", "alpha"]].round(3).to_string(index=False))
    print(f"[puntos de operación: {mejor}]")
    print(po.round(3).to_string(index=False))
    print(f"[listo] {time.time() - t0:.0f}s -> {OUT}")


if __name__ == "__main__":
    main()


## 2. Muestra con contexto de red completo (se crea solo la primera vez)

In [ ]:
USER_PCT = 5      # % de usuarios evaluados
RUTA_ZIP = ""     # opcional: ruta exacta del zip en tu Drive. Si la dejas vacía, se busca sola.

import os, time
from google.colab import drive
drive.mount("/content/drive")
RAIZ = "/content/drive/MyDrive"
CARPETA = f"{RAIZ}/AdaptAuth-ML"
os.makedirs(CARPETA, exist_ok=True)
MUESTRA_CTX = f"{CARPETA}/rba_muestra_usuarios_{USER_PCT}pct_contexto.parquet"

if os.path.exists(MUESTRA_CTX):
    print("Muestra con contexto encontrada en Drive:", MUESTRA_CTX)
else:
    if not RUTA_ZIP:
        candidatos = [os.path.join(r, f) for r, _, fs in os.walk(RAIZ) for f in fs
                      if "rba" in f.lower() and f.lower().endswith((".zip", ".csv"))]
        for c in candidatos:
            print(f"  encontrado: {c} ({os.path.getsize(c) / 1e9:.2f} GB)")
        if candidatos:
            RUTA_ZIP = max(candidatos, key=os.path.getsize)
        else:
            print("No está en tu Drive: descargando de Zenodo (solo esta vez)...")
            RUTA_ZIP = f"{CARPETA}/rba-dataset.zip"
            !wget -q --show-progress -O "{RUTA_ZIP}" "https://zenodo.org/records/6782156/files/rba-dataset.zip?download=1"
    print("Usando:", RUTA_ZIP, "| copiando al disco local de Colab para leerlo más rápido...")
    !cp "{RUTA_ZIP}" /content/rba_local.zip
    RUTA_ZIP = "/content/rba_local.zip"
    print("Recorriendo todo el dataset para el contexto de red (solo esta vez, 15 a 25 min)...")
    import adaptauth as aa
    t = time.time()
    aa.build_full_context_sample(RUTA_ZIP, USER_PCT).to_parquet(MUESTRA_CTX, index=False)
    print(f"Muestra guardada en Drive: {MUESTRA_CTX} ({(time.time() - t) / 60:.1f} min)")
!ls -lh "{CARPETA}"

## 3. Ejecutar el experimento

In [ ]:
!python run_experimento_v5.py --data "{MUESTRA_CTX}"

## 4. Resultados

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
r = json.load(open("results/metricas_v5.json"))
dg = json.load(open("results/diagnostico_etiquetas.json"))
print("Contexto de red:", r["contexto_red"], "| Volumen:", r["volumen"])
print("Mejor modelo según validación:", r["mejor_por_validacion"])
display(pd.read_csv("results/tabla_comparativa_v5.csv").round(4))
print("Puntos de operación (umbral fijado en validación, métricas en prueba):")
display(pd.read_csv("results/puntos_operacion_v5.csv").round(4))
for k in ["prueba_pareada_hibrido_vs_reglas", "prueba_pareada_hibrido_vs_iforest_v4", "prueba_pareada_hibrido_vs_iforest_v5"]:
    p = r[k]
    print(f"{k}: {p['prueba']}, estadístico={p['estadistico']:.2f}, p={p['p_valor']:.2e}, dz={p['d_cohen_dz']:.2f} | M_A={p['media_baseline']:.3f} M_B={p['media_adaptauth']:.3f}")
print("Latencia (ms):", {k: (round(v["p99_ms"], 2) if isinstance(v, dict) else round(v, 2)) for k, v in r["latencia_ms"].items()})
print("Importancia (supervisado):", list(r["importancia_supervisado_v5"].items())[:6])
print("SHAP IF v5:", r.get("shap_iforest_v5"))
print("\nDiagnóstico de la etiqueta:")
for k, v in dg.items():
    if k != "auc_univariada_v5_poblacion": print(f"  {k}: {v}")
display(pd.Series(dg["auc_univariada_v5_poblacion"], name="AUC univariada").sort_values())
print(pd.read_csv("results/matriz_3x2_hibrido_v5.csv", index_col=0))
for f in ["comparacion_modelos_v5", "puntos_operacion_v5", "f1_por_bloque_v5", "distribucion_riesgo_v5",
          "importancia_supervisado_v5", "shap_iforest_v5"]:
    if os.path.exists(f"results/figuras/{f}.png"):
        display(Image(f"results/figuras/{f}.png"))

## 5. Guardar en Google Drive
Quedan en **Mi unidad → AdaptAuth-ML → results_v5**. Mándame `metricas_v5.json` y `diagnostico_etiquetas.json`.

In [ ]:
!rm -rf "{CARPETA}/results_v5" && mkdir -p "{CARPETA}/results_v5" && cp -r results/* "{CARPETA}/results_v5/"
print("Resultados guardados en Mi unidad / AdaptAuth-ML / results_v5")